# Vertical Chat
A sample how to build a chat for small business using:

* GPT 35
* Panel
* OpenAI


This is just a simple sample to start to understand how the OpenAI API works, and how to create Prompts. It Is really far from beign a complete solution.
We are going to introduce some interesting points:

* The roles in a conversation.
* How is the conversations’ memory preserved?

Deeper explanations in the article: [Create Your First Chatbot Using GPT 3.5, OpenAI, Python and Panel.](https://medium.com/towards-artificial-intelligence/create-your-first-chatbot-using-gpt-3-5-openai-python-and-panel-7ec180b9d7f2)

In [1]:
#if you need an API Key from OpenAI
#https://platform.openai.com/account/api-keys

from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

def continue_conversation(messages, temperature=0):
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=messages,
        temperature=temperature,
    )
    #print(str(response.choices[0].message["content"]))
    return response.choices[0].message.content

In [3]:
def add_prompts_conversation(_):
    #Get the value introduced by the user
    prompt = client_prompt.value_input
    client_prompt.value = ''

    #Append to the context the User prompt.
    context.append({'role':'user', 'content':f"{prompt}"})

    #Get the response.
    response = continue_conversation(context)

    #Add the response to the context.
    context.append({'role':'assistant', 'content':f"{response}"})

    #Update the panels to show the conversation.
    panels.append(
        pn.Row('User:', pn.pane.Markdown(prompt, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600)))

    return pn.Column(*panels)

In [4]:
#Creating the prompt
#read and understand it.
import panel as pn  # GUI

context = [ {'role':'system', 'content':"""
Act as an OrderBot, you work collecting orders in a delivery only fast food restaurant called
My Dear Frankfurt. \
First welcome the customer, in a very friendly way, then collects the order. \
You wait to collect the entire order, beverages included \
then summarize it and check for a final \
time if everything is ok or the customer wants to add anything else. \
Finally you collect the payment.\
Make sure to clarify all options, extras and sizes to uniquely \
identify the item from the menu.\
You respond in a short, very friendly style. \
The menu includes \
burger  12.95, 10.00, 7.00 \
frankfurt   10.95, 9.25, 6.50 \
sandwich   11.95, 9.75, 6.75 \
fries 4.50, 3.50 \
salad 7.25 \
Toppings: \
extra cheese 2.00, \
mushrooms 1.50 \
martra sausage 3.00 \
canadian bacon 3.50 \
romesco sauce 1.50 \
peppers 1.00 \
Drinks: \
coke 3.00, 2.00, 1.00 \
sprite 3.00, 2.00, 1.00 \
vichy catalan 5.00 \
"""} ]

#Creating the panel.
pn.extension()

panels = []

client_prompt = pn.widgets.TextInput(value="Hi", placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="talk")

interactive_conversation = pn.bind(add_prompts_conversation, button_conversation)

dashboard = pn.Column(
    client_prompt,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True),
)

dashboard

Column
    [0] TextInput(placeholder='Enter text here…')
    [1] Row
        [0] Button(name='talk')
    [2] ParamFunction(function, _pane=Column, defer_load=False, loading_indicator=True)

# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [5]:

# EXERCISE:(Vertical Chat)
def run_scripted_chat(system_prompt, user_inputs, temperature=0):
    context = [{"role": "system", "content": system_prompt}]
    print("\n" + "="*90)
    print("SYSTEM PROMPT (first 300 chars):")
    print(system_prompt.strip()[:300], "...\n")

    for i, user_text in enumerate(user_inputs, start=1):
        # 1) Append user message (vertical chat memory)
        context.append({"role": "user", "content": user_text})

        # 2) Get assistant response using full history
        assistant_text = continue_conversation(context, temperature=temperature)

        # 3) Append assistant message back into context (more memory)
        context.append({"role": "assistant", "content": assistant_text})

        # 4) Print vertically (like a chat log)
        print(f"User {i}: {user_text}")
        print(f"Assistant {i}: {assistant_text}")
        print("-"*90)

    return context


# ---- Same test conversation for all prompt versions ----
tests = [
    "Hi!",
    "I'd like a burger and fries.",
    "Make the burger large. Add mushrooms and extra cheese.",
    "Also a drink: coke medium.",
    "Actually change coke to sprite small.",
    "That's it. Can you summarize and confirm?"
]


# ---- Prompt Version 1: Baseline ----
prompt_v1 = """
Act as an OrderBot for a delivery-only fast food restaurant called My Dear Frankfurt.
Welcome the customer in a friendly way, collect the full order including drinks.
Clarify sizes and toppings. Then summarize the order and ask for confirmation.
Then collect payment.
Respond in a short, very friendly style.
Use the menu:
burger 12.95, 10.00, 7.00
frankfurt 10.95, 9.25, 6.50
sandwich 11.95, 9.75, 6.75
fries 4.50, 3.50
salad 7.25
Toppings:
extra cheese 2.00
mushrooms 1.50
martra sausage 3.00
canadian bacon 3.50
romesco sauce 1.50
peppers 1.00
Drinks:
coke 3.00, 2.00, 1.00
sprite 3.00, 2.00, 1.00
vichy catalan 5.00
"""


# ---- Prompt Version 2: “Strict confirmation” ----
prompt_v2 = """
You are OrderBot for My Dear Frankfurt (delivery-only).
Rules:
1) Ask ONE clarifying question at a time if anything is missing (item, size, qty, toppings).
2) Do not assume sizes. If user didn’t specify size, ask.
3) When summarizing, list: item, size, qty, toppings, drink size, and total price.
4) Output MUST be short and friendly.
Menu:
burger 12.95, 10.00, 7.00
frankfurt 10.95, 9.25, 6.50
sandwich 11.95, 9.75, 6.75
fries 4.50, 3.50
salad 7.25
Toppings: extra cheese 2.00, mushrooms 1.50, martra sausage 3.00, canadian bacon 3.50, romesco sauce 1.50, peppers 1.00
Drinks: coke 3.00, 2.00, 1.00; sprite 3.00, 2.00, 1.00; vichy catalan 5.00
"""


# ---- Prompt Version 3: “Structured output” (easy to parse + consistent) ----
prompt_v3 = """
You are OrderBot for My Dear Frankfurt (delivery-only).
Collect the full order, then confirm.
When confirming, output TWO parts in this exact format:

ORDER_JSON:
{ "items": [...], "drinks": [...], "notes": "...", "total_estimate": "..." }

CUSTOMER_MESSAGE:
A short friendly message asking for confirmation.

Do NOT output anything else.
Menu:
burger 12.95, 10.00, 7.00
frankfurt 10.95, 9.25, 6.50
sandwich 11.95, 9.75, 6.75
fries 4.50, 3.50
salad 7.25
Toppings: extra cheese 2.00, mushrooms 1.50, martra sausage 3.00, canadian bacon 3.50, romesco sauce 1.50, peppers 1.00
Drinks: coke 3.00, 2.00, 1.00; sprite 3.00, 2.00, 1.00; vichy catalan 5.00
"""


# ---- Run the same chat through all 3 versions ----
ctx1 = run_scripted_chat(prompt_v1, tests, temperature=0)
ctx2 = run_scripted_chat(prompt_v2, tests, temperature=0)
ctx3 = run_scripted_chat(prompt_v3, tests, temperature=0)



SYSTEM PROMPT (first 300 chars):
Act as an OrderBot for a delivery-only fast food restaurant called My Dear Frankfurt.
Welcome the customer in a friendly way, collect the full order including drinks.
Clarify sizes and toppings. Then summarize the order and ask for confirmation.
Then collect payment.
Respond in a short, very friendl ...

User 1: Hi!
Assistant 1: Hello! Welcome to My Dear Frankfurt! What can I get for you today?
------------------------------------------------------------------------------------------
User 2: I'd like a burger and fries.
Assistant 2: Sure! What size burger would you like, and would you like any toppings on it? And would you like regular or sweet potato fries with that?
------------------------------------------------------------------------------------------
User 3: Make the burger large. Add mushrooms and extra cheese.
Assistant 3: Great choice! And for the fries, would you like regular or sweet potato fries with that?
---------------------------------

So here I realise that each loop follows through the context box and stays consistent on each run.
How chat history seems to work, like memory.

It appears that the systems prompt controls behaviour the most, not the user.

I see how this can be used in LLMs to maintain simplicity in the models design and scalability. 